In [0]:
%sql
CREATE OR REPLACE TABLE cpt_utility_catalog.gold.fct_substation_daily
AS
WITH daily_telemetry_summary AS (

    SELECT
        CAST(date_format(s.timestamp, 'yyyyMMdd') AS INT) AS raw_date_key,
        LOWER(TRIM(s.substation))                         AS clean_substation_name,
        
        CAST(AVG(s.load_MVA) AS DECIMAL(13,9))            AS mean_load_MVA,
        CAST(MAX(s.load_MVA) AS DECIMAL(13,9))            AS peak_load_MVA,
        CAST(MIN(s.load_MVA) AS DECIMAL(13,9))            AS min_load_MVA
        
    FROM cpt_utility_catalog.silver.silver_substations_cleaned s
    WHERE s.timestamp IS NOT NULL
    GROUP BY 
        CAST(date_format(s.timestamp, 'yyyyMMdd') AS INT),
        LOWER(TRIM(s.substation))
)

SELECT

    xxhash64(
        CONCAT_WS('||', 
            CAST(COALESCE(a.raw_date_key, -1) AS STRING), 
            CAST(COALESCE(ds.substation_key, xxhash64('unmapped')) AS STRING)
        )
    ) AS sub_daily_key,

    COALESCE(a.raw_date_key, -1)                      AS date_key,
    COALESCE(ds.substation_key, xxhash64('unmapped')) AS substation_key,

    a.mean_load_MVA,
    a.peak_load_MVA,
    a.min_load_MVA

FROM daily_telemetry_summary a

LEFT JOIN cpt_utility_catalog.gold.dim_substation ds
       ON xxhash64(a.clean_substation_name) = ds.substation_key;